<img src="http://imgur.com/1ZcRyrc.png" style="float: left; margin-right: 20px; height: 55px" height="55px">

# 3. Retrieval & Reranking Techniques (Solved Reference)

Fully worked solutions to all three exercises. Use this to check your work or to
catch up if you fell behind during the live lab. See `solutions folder` for
prose explanations alongside this code.

---

## Solution Guide

1. [Exercise 1 — Negation-Sensitive Cross-Encoder Scoring](#exercise1)
2. [Exercise 2 — Weighted MaxSim for Late Interaction](#exercise2)
3. [Exercise 3 — Sweeping `nlist` and `nprobe` Together](#exercise3)

---

In [36]:
 %pip install faiss-cpu numpy langchain langchain-community langchain-cohere langchain-huggingface flashrank rank_bm25 pylate

In [ ]:
!mkdir -p ./data

In [44]:
import os


# URLs for the files
capstone_corpus_url = "https://raw.githubusercontent.com/ga-curriculum-dev/Ai4-Agentic-Ops---Retrieval-Augmented-Generation--RAG--in-Practice-4-hrs/main/data/capstone_corpus.json"
golden_dataset_url = "https://raw.githubusercontent.com/ga-curriculum-dev/Ai4-Agentic-Ops---Retrieval-Augmented-Generation--RAG--in-Practice-4-hrs/main/data/golden_dataset_module4.json"

# Download capstone_corpus.json
print(f"Downloading {os.path.basename(capstone_corpus_url)}...")
!wget -O ./data/capstone_corpus.json $capstone_corpus_url

# Download golden_dataset_module4.json
print(f"Downloading {os.path.basename(golden_dataset_url)}...")
!wget -O ./data/golden_dataset_module4.json $golden_dataset_url

print("Files downloaded to ./data/")

--2026-07-31 17:27:38--  https://raw.githubusercontent.com/ga-curriculum-dev/Ai4-Agentic-Ops---Retrieval-Augmented-Generation--RAG--in-Practice-4-hrs/main/data/capstone_corpus.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-07-31 17:27:38 ERROR 404: Not Found.

--2026-07-31 17:27:38--  https://raw.githubusercontent.com/ga-curriculum-dev/Ai4-Agentic-Ops---Retrieval-Augmented-Generation--RAG--in-Practice-4-hrs/main/data/golden_dataset_module4.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-07-31 17:27:38 ERROR 404:

In [41]:
import os
import getpass

# Set your Cohere API Key here if you have one. This is optional.
cohere_key = getpass.getpass("Enter your Cohere API Key (leave empty to skip): ")
if cohere_key:
    os.environ["CO_API_KEY"] = cohere_key

print("API key input prompts added.")

Enter your Cohere API Key (leave empty to skip): ··········
API key input prompts added.


<h2 id="exercise1"> Exercise 1 — Negation-Sensitive Cross-Encoder Scoring </h2>


In [2]:
SPECIFIC_TERMS = {"file": 1.0, "upload": 1.0, "uploading": 1.0, "2gb": 1.0, "large": 0.7}
GENERIC_TERMS  = {"crash": 0.3, "crashes": 0.3, "app": 0.2, "startup": 0.4}
STOPWORDS = {"the", "a", "an", "on", "in", "to", "of", "and", "causes", "after", "than", "for"}
NEGATION_WORDS = {"not", "no", "never", "doesn't", "don't", "won't", "isn't"}
NEGATION_WINDOW = 6

def normalize(word: str) -> str:
    return word.rstrip(".,").lower()

def lemma(word: str) -> str:
    for suffix in ("ing", "es", "s"):
        if word.endswith(suffix) and len(word) > len(suffix) + 2:
            return word[: -len(suffix)]
    return word

def find_negated_lemmas(doc: str) -> set:
    words = [normalize(w) for w in doc.split()]
    negated = set()
    for i, w in enumerate(words):
        if w in NEGATION_WORDS:
            for j in range(i + 1, min(i + 1 + NEGATION_WINDOW, len(words))):
                negated.add(lemma(words[j]))
    return negated

def cross_encoder_score_v2(query: str, doc: str) -> float:
    q_words = [normalize(w) for w in query.split() if normalize(w) not in STOPWORDS]
    d_lemmas = {lemma(normalize(w)) for w in doc.split()}
    negated_lemmas = find_negated_lemmas(doc)

    score, total_weight = 0.0, 0.0
    for w in q_words:
        weight = SPECIFIC_TERMS.get(w, GENERIC_TERMS.get(w, 0.1))
        total_weight += weight
        w_lemma = lemma(w)
        if w_lemma in d_lemmas:
            if w_lemma in negated_lemmas:
                score -= weight * 0.5
            else:
                score += weight
    return score / total_weight if total_weight else 0.0


In [3]:
query = "app crashes on large file upload"
docs = [
    ("true_positive", "Uploading a file larger than 2GB causes the app to crash immediately."),
    ("negated_false_positive", "Large file uploads do not cause the app to crash in the latest release."),
    ("offtopic", "General troubleshooting steps for slow app performance."),
]

for name, doc in docs:
    print(f"{cross_encoder_score_v2(query, doc):.3f}  [{name}] {doc}")

assert cross_encoder_score_v2(query, docs[0][1]) > cross_encoder_score_v2(query, docs[1][1]), \
    "True positive should outrank the negated false positive"
print("\nConfirmed: true positive ranks above the negated (false-positive) match.")


0.781  [true_positive] Uploading a file larger than 2GB causes the app to crash immediately.
0.766  [negated_false_positive] Large file uploads do not cause the app to crash in the latest release.
0.062  [offtopic] General troubleshooting steps for slow app performance.

Confirmed: true positive ranks above the negated (false-positive) match.


<h2 id="exercise2"> Exercise 2 — Weighted MaxSim for Late Interaction </h2>

In [4]:
import re
import hashlib
import numpy as np

DIM = 32

def token_vec(word: str, dim: int = DIM) -> np.ndarray:
    h = int(hashlib.md5(word.encode()).hexdigest(), 16)
    rng = np.random.RandomState(h % (2**32))
    v = rng.normal(size=dim)
    return v / np.linalg.norm(v)

def tokenize(text: str) -> list:
    return text.lower().replace(",", "").replace(".", "").split()

def colbert_maxsim(query: str, doc: str) -> float:
    q_vecs = np.array([token_vec(w) for w in tokenize(query)])
    d_vecs = np.array([token_vec(w) for w in tokenize(doc)])
    sims = q_vecs @ d_vecs.T
    return float(sims.max(axis=1).sum())

ALPHANUMERIC_ID_PATTERN = re.compile(r"^[a-z]\d+$|^\d+[a-z]+$", re.IGNORECASE)

def weighted_colbert_maxsim(query: str, doc: str, high_weight: float = 3.0, low_weight: float = 1.0) -> float:
    q_words = tokenize(query)
    q_vecs = np.array([token_vec(w) for w in q_words])
    d_vecs = np.array([token_vec(w) for w in tokenize(doc)])
    sims = q_vecs @ d_vecs.T
    max_per_token = sims.max(axis=1)
    weights = np.array([high_weight if ALPHANUMERIC_ID_PATTERN.match(w) else low_weight for w in q_words])
    return float((max_per_token * weights).sum())


In [5]:
query = "M8 bolt torque spec actuator housing"
doc_specific = "Torque the M8 fastener to spec before reassembly."
doc_generic  = "Replace the bolt securing the outer housing spec panel."

print("Plain MaxSim:")
print(f"  doc_specific: {colbert_maxsim(query, doc_specific):.3f}")
print(f"  doc_generic:  {colbert_maxsim(query, doc_generic):.3f}")

print("Weighted MaxSim (alphanumeric IDs weighted higher):")
print(f"  doc_specific: {weighted_colbert_maxsim(query, doc_specific):.3f}")
print(f"  doc_generic:  {weighted_colbert_maxsim(query, doc_generic):.3f}")

assert colbert_maxsim(query, doc_generic) > colbert_maxsim(query, doc_specific), \
    "Plain MaxSim should (incorrectly) favor the generic document"
assert weighted_colbert_maxsim(query, doc_specific) > weighted_colbert_maxsim(query, doc_generic), \
    "Weighted MaxSim should correctly favor the specific document"
print("\nConfirmed: plain MaxSim favors the generic doc; weighted MaxSim correctly favors the specific doc.")


Plain MaxSim:
  doc_specific: 3.948
  doc_generic:  4.119
Weighted MaxSim (alphanumeric IDs weighted higher):
  doc_specific: 5.948
  doc_generic:  5.170

Confirmed: plain MaxSim favors the generic doc; weighted MaxSim correctly favors the specific doc.


<h2 id="exercise3"> Exercise 3 — Sweeping `nlist` and `nprobe` Together </h2>

In [6]:
import faiss
import numpy as np
import time

def find_min_nprobe_for_recall(nlist: int, target_recall: float, vectors, queries, flat_ids, dim: int) -> dict:
    quantizer = faiss.IndexFlatL2(dim)
    index = faiss.IndexIVFFlat(quantizer, dim, nlist)
    index.train(vectors)
    index.add(vectors)

    for nprobe in [1, 2, 4, 8, 16, 32, 64, 128]:
        if nprobe > nlist:
            break
        index.nprobe = nprobe

        # take the minimum over several timed repeats to reduce system jitter
        best_ms = float("inf")
        for _ in range(7):
            t0 = time.perf_counter()
            _, ids = index.search(queries, k=10)
            elapsed_ms = (time.perf_counter() - t0) * 1000 / len(queries)
            best_ms = min(best_ms, elapsed_ms)

        recalls = [len(set(flat_ids[i]) & set(ids[i])) / 10.0 for i in range(len(queries))]
        avg_recall = sum(recalls) / len(recalls)
        if avg_recall >= target_recall:
            return {"nlist": nlist, "nprobe": nprobe, "latency_ms": best_ms, "recall": avg_recall}
    return {"nlist": nlist, "nprobe": None, "latency_ms": None, "recall": None}


In [7]:
np.random.seed(7)
dim = 128
num_vectors = 100_000
num_clusters = 200
centers = np.random.random((num_clusters, dim)).astype("float32") * 10
labels = np.random.randint(0, num_clusters, size=num_vectors)
vectors = centers[labels] + np.random.normal(scale=0.5, size=(num_vectors, dim)).astype("float32")
queries = centers[np.random.randint(0, num_clusters, size=50)] + \
          np.random.normal(scale=0.5, size=(50, dim)).astype("float32")

flat_index = faiss.IndexFlatL2(dim)
flat_index.add(vectors)
_, flat_ids = flat_index.search(queries, k=10)

print(f"{'nlist':<10}{'min nprobe for 95% recall':<28}{'latency (ms)':<15}{'recall'}")
results = []
for nlist in [100, 400, 800]:
    r = find_min_nprobe_for_recall(nlist, 0.95, vectors, queries, flat_ids, dim)
    results.append(r)
    print(f"{r['nlist']:<10}{str(r['nprobe']):<28}{r['latency_ms']:<15.4f}{r['recall']:.0%}")

best = min([r for r in results if r["nprobe"] is not None], key=lambda r: r["latency_ms"])
print(f"\nLowest latency at 95%+ recall: nlist={best['nlist']}, nprobe={best['nprobe']}, {best['latency_ms']:.4f}ms")


nlist     min nprobe for 95% recall   latency (ms)   recall
100       1                           0.0976         100%
400       4                           0.0730         100%
800       4                           0.0478         97%

Lowest latency at 95%+ recall: nlist=800, nprobe=4, 0.0478ms


# Extensions: Local Models vs. Cloud/API Models

> **The code cells below require either a paid API key (Cloud/API track) or a local model download**

## Extension A — Local Models Approach

**Retrieval & Reranking Techniques**

The course simulates cross-encoder and ColBERT scoring with transparent term-weighting
functions specifically because real pretrained reranker weights require a network
download this course's sandbox couldn't reach at all. This section shows the real
*local* tooling equivalents.

### What Changes, Component by Component

| Course concept | Course's stand-in | Local-model equivalent |
|---|---|---|
| Hybrid search (Objective 1 groundwork) | TF-IDF lexical retrieval | `langchain_community.retrievers.BM25Retriever` (sparse) + `FAISS` (dense) |
| Fusion | Reciprocal Rank Fusion, hand-implemented | `langchain.retrievers.EnsembleRetriever` (RRF built in) |
| Cross-encoder reranking | `cross_encoder_score()` term-weighting function | `flashrank` (`Ranker`, `RerankRequest`) — a genuinely lightweight, local cross-encoder library |
| Late interaction / ColBERT | `colbert_maxsim()` hash-vector MaxSim | `pylate` or a local ColBERT checkpoint (heavier setup, see note below) |
| Index tuning | Real FAISS (already used as-is) | Same — no change needed |

### 1. Hybrid Search with Real Local Retrievers

In [34]:
import json

with open('./data/capstone_corpus.json', 'r') as f:
    corpus = json.load(f)

corpus_texts = [d.get('text') for d in corpus.get('knowledge_base')]


In [37]:
# -- requires `langchain-community` and `langchain`
from langchain_community.retrievers import BM25Retriever
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.retrievers import EnsembleRetriever

sparse_retriever = BM25Retriever.from_texts(corpus_texts)
sparse_retriever.k = 20
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
dense_retriever = FAISS.from_texts(corpus_texts, embeddings).as_retriever(search_kwargs={"k": 20})

# EnsembleRetriever implements Reciprocal Rank Fusion internally --
# this replaces the course's hand-written reciprocal_rank_fusion() function.
hybrid_retriever = EnsembleRetriever(
    retrievers=[sparse_retriever, dense_retriever],
    weights=[0.5, 0.5],
)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [38]:
hybrid_retriever.invoke('pto policy')

[Document(metadata={}, page_content='The Initech handbook states that travel policy follows regional guidelines set by Operations.'),
 Document(metadata={}, page_content='The Aperture Labs handbook states that travel policy follows regional guidelines set by Sales.'),
 Document(metadata={}, page_content='For Umbrella employees, travel policy is managed by the Facilities team and reviewed on a quarterly basis.'),
 Document(metadata={}, page_content='The Umbrella handbook states that travel policy follows regional guidelines set by Sales.'),
 Document(metadata={}, page_content='For Initech employees, travel policy is managed by the Legal team and reviewed on a quarterly basis.'),
 Document(metadata={}, page_content='For Soylent Corp employees, travel policy is managed by the Operations team and reviewed on a quarterly basis.'),
 Document(metadata={}, page_content='The Wonka Industries handbook states that travel policy follows regional guidelines set by Sales.'),
 Document(metadata={}, p

### 2. Cross-Encoder Reranking with FlashRank

FlashRank is worth calling out specifically: it ships small, fast, ONNX-optimized
cross-encoder models that run comfortably on CPU with no GPU requirement — a genuinely
good fit for a "local, no API" constraint.

In [20]:
# OPTIONAL / ILLUSTRATIVE -- requires `flashrank` and a one-time model download (~35MB); not executed here.
from flashrank import Ranker, RerankRequest

ranker = Ranker(model_name="ms-marco-MiniLM-L-12-v2")  # downloads once (~35MB), then fully offline

def rerank_local(query: str, candidates: list) -> list:
    request = RerankRequest(query=query, passages=[{"text": c} for c in candidates])
    return ranker.rerank(request)  # returns candidates sorted by cross-encoder score


ms-marco-MiniLM-L-12-v2.zip: 100%|██████████| 21.6M/21.6M [00:00<00:00, 37.6MiB/s]


Or via the LangChain compressor wrapper, which composes directly with
`hybrid_retriever` above:

In [40]:
# OPTIONAL / ILLUSTRATIVE -- requires `langchain`, `flashrank`, and `hybrid_retriever` from above; not executed here.
from langchain_classic.retrievers.document_compressors import FlashrankRerank
from langchain_classic.retrievers import ContextualCompressionRetriever

compressor = FlashrankRerank(model="ms-marco-MiniLM-L-12-v2", top_n=5)
reranked_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=hybrid_retriever,
)


### 3. Late Interaction (ColBERT) — the Heaviest Local Ask in This Module

Real ColBERT / PyLate models are meaningfully heavier to self-host than FlashRank's
small cross-encoders — expect a larger model download and a real GPU benefit for
acceptable latency at scale. For most take-home purposes, FlashRank's cross-encoder
reranking already captures most of the practical precision gain the course's Objective
2 demonstrates; only reach for a real local ColBERT setup if you're specifically
validating the long-document dilution behavior Lesson 3 demonstrates (where MaxSim's
per-token scoring resists the "buried in a long document" problem that both bi-encoders
and even cross-encoders can struggle with at extreme document lengths).

### What This Buys You, and What It Costs

**Pros:**
- FlashRank in particular is close to a free lunch: small download, CPU-friendly, no API key, no rate limits — a strong default for a local hybrid+rerank stack
- `EnsembleRetriever`'s built-in RRF removes the need to maintain the course's hand-written fusion function, once you trust the library implementation
- Fully offline after the one-time model downloads

**Cons:**
- A real ColBERT/PyLate setup is a genuinely heavier local dependency than anything else in this course's local track — budget real setup time and, ideally, GPU access
- `EnsembleRetriever`'s RRF is a black box relative to the course's from-scratch implementation — you lose the ability to easily inspect or modify the `k` smoothing constant the course's exercise has you tune directly
- Local cross-encoder quality (FlashRank's small models) trails the largest hosted rerankers on harder discrimination tasks, though the gap is smaller here than for generation

### Bridging Back to the Course

Lesson 3's central lessons — why cross-encoders outperform bi-encoders on precision,
why reranking must be a second stage rather than the first, and why MaxSim resists
document-length dilution — all hold with these real components. The `nprobe`/`nlist`
tuning exercise is unaffected either way, since FAISS itself doesn't
change.

---

## Extension B — Cloud/API Models Approach

**Retrieval & Reranking Techniques**

This section shows the real *cloud/API* tooling equivalents for hybrid search and
reranking.

### What Changes, Component by Component

| Course concept | Course's stand-in | Cloud/API equivalent |
|---|---|---|
| Hybrid search (Objective 1 groundwork) | TF-IDF lexical retrieval | `BM25Retriever` (still local — see note) + `FAISS`/`OpenAIEmbeddings` for dense |
| Fusion | Reciprocal Rank Fusion, hand-implemented | `EnsembleRetriever`, or a managed hybrid search service (Azure AI Search, Elasticsearch) |
| Cross-encoder reranking | `cross_encoder_score()` term-weighting function | `langchain_cohere.CohereRerank` (Cohere's hosted Rerank API) |
| Late interaction / ColBERT | `colbert_maxsim()` hash-vector MaxSim | No mainstream hosted-API equivalent — see note below |
| Index tuning | Real FAISS (already used as-is) | Same, or delegated to a managed vector store's internal tuning |

### 1. Sparse Retrieval Has No Real Cloud Equivalent — And That's Fine

BM25 is a statistical, corpus-local algorithm; there isn't a standard hosted
"BM25-as-a-service" product most teams reach for. Keep `BM25Retriever` local even in an
otherwise cloud-first architecture — it's genuinely fine, and inexpensive, to run
locally alongside a cloud dense retriever. The exception is if you adopt a fully
managed hybrid search product (Azure AI Search, Elasticsearch/OpenSearch with a managed
hosting tier) that bundles sparse and dense retrieval together — a valid but heavier
architectural commitment than swapping one component.

### 2. Cloud Cross-Encoder Reranking via Cohere

Cohere's Rerank API is the most widely used **hosted** cross-encoder reranking product
— a direct cloud analog to FlashRank's local models:

In [42]:
# OPTIONAL / ILLUSTRATIVE -- requires `langchain-cohere` and a funded Cohere API key; not executed here.
from langchain_cohere import CohereRerank
from langchain_classic.retrievers import ContextualCompressionRetriever

compressor = CohereRerank(model="rerank-english-v3.0", top_n=5)
reranked_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=hybrid_retriever,   # e.g. an EnsembleRetriever combining BM25 + a cloud-embedded FAISS index
)


In [43]:
reranked_retriever.invoke('pto policy')

[Document(metadata={'relevance_score': 0.99686414}, page_content='2019 PTO Policy: Full-time employees accrue 15 days of paid time off per year, credited monthly.'),
 Document(metadata={'relevance_score': 0.9914225}, page_content='2024 PTO Policy Addendum (supersedes the 2019 policy): Full-time employees now receive unlimited PTO, subject to manager approval and a minimum of 10 days taken per year.'),
 Document(metadata={'relevance_score': 0.10158945}, page_content='Stark Industries Finance policy: travel policy requests must be submitted through the internal portal at least 5 business days in advance.'),
 Document(metadata={'relevance_score': 0.0570714}, page_content='Cyberdyne Finance policy: travel policy requests must be submitted through the internal portal at least 5 business days in advance.'),
 Document(metadata={'relevance_score': 0.043284845}, page_content='Umbrella Operations policy: travel policy requests must be submitted through the internal portal at least 5 business day

**Trade-off:** this gives you a state-of-the-art reranker with no model to download
or host, at the cost of one network call per rerank operation — and rerank calls
happen on the *candidate set*, not just once per query, so this is a meaningfully
higher-volume API dependency than the router or embeddings calls in Modules 1–2.


### What This Buys You, and What It Costs

**Pros:**
- Cohere's hosted reranker is a strong, well-maintained, state-of-the-art cross-encoder with zero local model management
- No GPU requirement, no model download, works identically across every developer's machine

**Cons — and these compound with Lesson 3's own architecture lesson:**
- Reranking runs on every retrieved candidate for every query — at real production query volume, this is a much higher API call count than routing (Lesson 1) or embedding (Lesson 2), and the associated cost and rate-limit exposure scale accordingly. This is exactly why Lesson 3 teaches reranking as a precision-focused *second stage* applied only to a pre-narrowed candidate set (typically 20-50 documents) rather than the full corpus — that architectural discipline matters *more*, not less, once each rerank call has a real API cost attached.
- Network latency is now a real, variable addition to the rerank stage's slice of the latency budget (Lesson 1) — profile it for real rather than trusting the course's simulated ~150ms rerank budget.
- No mainstream hosted equivalent for BM25 or ColBERT means a "fully cloud" architecture for this module is, in practice, a hybrid of local (BM25, optionally ColBERT) and cloud (dense embeddings, cross-encoder rerank) components — worth setting that expectation up front rather than assuming everything in this module has a drop-in cloud swap.

### Bridging Back to the Course

The core lesson that reranking must be gated to a small candidate set because its cost
scales with candidates, not corpus size, is *more* consequential with a real hosted
reranker than with the course's free local function — the course's architectural
discipline pays for itself literally, not just conceptually, once you're paying per
rerank call.